# Notebook 02 — Calidad, limpieza y preparación

**Objetivo:** Preparar el dataset a partir de decisiones justificadas con evidencia observada en la inspección inicial. Cada decisión indica la evidencia que la motivó, la acción aplicada y el impacto observado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw/reporte_clinica.csv')
print(f'Dataset cargado. Dimensiones iniciales: {df.shape}')

## Paso 1 — Eliminación de columna de índice redundante

In [ ]:
# Evidencia: columna 'Unnamed: 0' contiene valores 0..1362, idénticos al índice de pandas.
# No aporta información analítica.
# Acción: eliminarla.
df = df.drop(columns=['Unnamed: 0'])
print('Columna Unnamed:0 eliminada. Columnas restantes:', df.columns.tolist())

## Paso 2 — Normalización de variables categóricas

In [ ]:
# Evidencia: en la inspección se detectaron categorías duplicadas por diferencias
# de capitalización en 'sex' y 'smoker', y abreviaciones mezcladas en 'region'.
# Sin normalización, los análisis agrupados producirían resultados incorrectos.

print('Antes de normalizar:')
print('sex:', df['sex'].unique())
print('smoker:', df['smoker'].unique())
print('region:', df['region'].unique())

df['sex'] = df['sex'].str.lower().str.strip()
df['smoker'] = df['smoker'].str.lower().str.strip()
region_map = {'SE': 'southeast', 'NE': 'northeast', 'SW': 'southwest', 'NW': 'northwest'}
df['region'] = df['region'].replace(region_map)

print('\nDespués de normalizar:')
print('sex:', df['sex'].unique())
print('smoker:', df['smoker'].unique())
print('region:', df['region'].unique())

**Decisión justificada:** La normalización de capitalización y el mapeo de abreviaciones no modifica la cantidad de registros ni elimina información. El impacto es que cada variable categórica queda con la cardinalidad correcta: `sex` con 2 categorías, `smoker` con 2, `region` con 4.

## Paso 3 — Tratamiento de outliers erróneos en `bmi`

In [ ]:
# Evidencia: se detectaron 6 registros con bmi=999.0, valor biológicamente imposible.
# El rango fisiológico del IMC se ubica aproximadamente entre 10 y 70.
# Decisión: eliminar los registros con bmi >= 100 ya que no pueden imputarse
# (el valor es claramente un error de carga, no un dato faltante).

n_antes = len(df)
outliers_bmi = df[df['bmi'] >= 100]
print(f'Registros con bmi >= 100: {len(outliers_bmi)}')
print(outliers_bmi[['bmi']].value_counts())

df = df[df['bmi'] < 100]
print(f'\nFilas eliminadas: {n_antes - len(df)}')
print(f'Filas restantes: {len(df)}')

## Paso 4 — Tratamiento de outliers erróneos en `children`

In [ ]:
# Evidencia: se detectaron valores -1 (imposibles) y 99 (inverosímil para un registro
# de seguro médico estándar). Estos valores no pueden imputarse ya que representan
# errores de carga, no ausencias de información.
# Decisión: eliminar registros con children < 0 o children > 20.

n_antes = len(df)
print('Distribución de children antes:')
print(df['children'].value_counts().sort_index())

df = df[(df['children'] >= 0) & (df['children'] <= 20)]
print(f'\nFilas eliminadas: {n_antes - len(df)}')
print(f'Filas restantes: {len(df)}')

## Paso 5 — Imputación de valores nulos en `age`

In [ ]:
# Evidencia: 102 registros (7.5%) tienen age nulo. Eliminarlos representaría
# una pérdida significativa de información. La distribución de age es
# aproximadamente simétrica, por lo que la mediana es un estimador robusto.
# Decisión: imputar con la mediana de la columna.

mediana_age = df['age'].median()
print(f'Nulos en age antes: {df["age"].isnull().sum()}')
print(f'Mediana de age: {mediana_age}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df['age'].dropna().hist(ax=axes[0], bins=20, color='steelblue')
axes[0].set_title('Distribución de age (sin nulos)')
axes[0].axvline(mediana_age, color='red', linestyle='--', label=f'Mediana: {mediana_age}')
axes[0].legend()

df['age'] = df['age'].fillna(mediana_age)
df['age'].hist(ax=axes[1], bins=20, color='steelblue')
axes[1].set_title('Distribución de age (tras imputación)')
plt.tight_layout()
plt.show()

print(f'Nulos en age después: {df["age"].isnull().sum()}')

## Paso 6 — Imputación de valores nulos en `bmi`

In [ ]:
# Evidencia: tras eliminar los outliers de bmi, quedan 44 registros con bmi nulo (3.4%).
# La distribución del IMC es aproximadamente normal en la población adulta,
# lo que hace a la mediana un estimador apropiado.
# Decisión: imputar con la mediana.

mediana_bmi = df['bmi'].median()
print(f'Nulos en bmi antes: {df["bmi"].isnull().sum()}')
print(f'Mediana de bmi: {mediana_bmi}')

df['bmi'] = df['bmi'].fillna(mediana_bmi)
print(f'Nulos en bmi después: {df["bmi"].isnull().sum()}')

## Paso 7 — Verificación final y guardado

In [ ]:
print('=== Estado final del dataset ===')
print(f'Dimensiones: {df.shape}')
print(f'Retención: {len(df)/1363*100:.1f}% de los registros originales')
print()
print('Nulos restantes:')
print(df.isnull().sum())
print()
print('Estadísticas descriptivas finales:')
df.describe()

In [ ]:
df.to_csv('../data/processed/reporte_clinica_clean.csv', index=False)
print('Dataset procesado guardado en data/processed/reporte_clinica_clean.csv')